In [1]:
import pandas as pd
import numpy as np

TARGETS = ["DeltaTheta", "DeltaX", "DeltaY"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [2]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [3]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
print(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====
        Target  Dataset           Best_Model      Neurons  \
0   DeltaTheta  Train_1           model_20_6           20   
1       DeltaX  Train_1     model_[10, 25]_7     [10, 25]   
2       DeltaY  Train_1   model_[2, 5, 10]_3   [2, 5, 10]   
3   DeltaTheta  Train_2  model_[5, 10, 15]_7  [5, 10, 15]   
4       DeltaX  Train_2   model_[2, 5, 10]_3   [2, 5, 10]   
5       DeltaY  Train_2  model_[5, 10, 15]_9  [5, 10, 15]   
6   DeltaTheta      Val       model_[2, 5]_1       [2, 5]   
7       DeltaX      Val   model_[2, 5, 10]_0   [2, 5, 10]   
8       DeltaY      Val   model_[2, 5, 10]_0   [2, 5, 10]   
9   DeltaTheta   Test_1       model_[2, 5]_6       [2, 5]   
10      DeltaX   Test_1  model_[5, 10, 15]_0  [5, 10, 15]   
11      DeltaY   Test_1   model_[2, 5, 10]_3   [2, 5, 10]   
12  DeltaTheta   Test_2      model_[5, 15]_9      [5, 15]   
13      DeltaX   Test_2   model_[2, 5, 10]_3   [2, 5, 10]   
14      DeltaY   Test_2  model_[5, 10, 